In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, ElasticNet, PoissonRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

CONFIG = {
    "file": "https://raw.githubusercontent.com/MeteoFur/ANHSF-Model/refs/heads/main/mainhurricanedatafile2.csv",
    "a": ["ENSO_MAM", "MDR_MAM", "ATL_MAM"],
    "b": ["named_storms", "hurricanes", "majors"],
    "k": 5,
    "mode": "normal",
    "vals": {
        "ENSO_MAM": float(input("Enter current ENSO MAM value: ")),
        "MDR_MAM": float(input("Enter current MDR MAM value: ")),
        "ATL_MAM": float(input("Enter current ATL MAM value: "))}}

df = pd.read_csv(CONFIG["file"])

cols = ["year"] + CONFIG["a"] + CONFIG["b"]

years = df["year"].to_numpy()
X = df[CONFIG["a"]].to_numpy(dtype=float)
Y = df[CONFIG["b"]].to_numpy(dtype=float)

print("\n",df.head())

if (X.shape == Y.shape) and (len(years) == X.shape[0]):
    print("\ndata file looks good :3")
else:
    print("SOMTHING IS WRONG WITH THE DATA FILE!")

In [ ]:
def analog(X_train, y_train, x_target, k=5, return_analogs=False):
    stsc = StandardScaler() # DONT REMOVE! idk why or how, but if you remove this line EVERYTHING breaks
    Xs = stsc.fit_transform(X_train)
    xt = stsc.transform(np.asarray(x_target).reshape(1, -1))[0]

    dists = np.linalg.norm(Xs - xt, axis=1)
    idx = np.argsort(dists)[:k]

    weights = 1.0 / (dists[idx] + 1e-6)
    weights = weights / weights.sum()

    pred = np.sum(y_train[idx] * weights[:, None], axis=0)

    if return_analogs:
        return pred, idx, dists[idx]

    return pred

In [ ]:
def train(X_train, Y_train):
    mod = {}

    mod["ridge"] = Ridge(alpha=1.0)
    mod["ridge"].fit(X_train, Y_train)

    mod["lasso"] = Lasso(alpha=0.05, max_iter=10000)
    mod["lasso"].fit(X_train, Y_train)

    mod["elastic"] = ElasticNet(alpha=0.05, l1_ratio=0.5, max_iter=10000)
    mod["elastic"].fit(X_train, Y_train)

    poisson_models = []
    for i in range(Y_train.shape[1]):
        y_col = Y_train[:, i]
        y_col = np.clip(y_col, 0, None)

        pm = PoissonRegressor(alpha=0.1, max_iter=1000)
        pm.fit(X_train, y_col)
        poisson_models.append(pm)

    mod["poisson"] = poisson_models
    return mod

In [ ]:
def reg(mod, x):
    x = np.asarray(x).reshape(1, -1)
    preds = {}

    for name, model in mod.items():
        if name == "poisson":
            preds[name] = np.array([m.predict(x)[0] for m in model])
        else:
            pred = model.predict(x)[0]
            preds[name] = np.clip(np.asarray(pred, dtype=float), 0, None)

    return preds

In [ ]:
def climatemodel(Y_train):
    return np.mean(Y_train, axis=0)

In [ ]:
def run_hindcast():
    rows = []

    for i in range(len(X)):
        if CONFIG["mode"] == "normal":
            train_idx = np.arange(i)
        elif CONFIG["mode"] == "testing":
            train_idx = np.delete(np.arange(len(X)), i)
        else:
            raise ValueError("mode must be set to 'normal' or 'testing'")
        if len(train_idx) < max(10, CONFIG["k"] + 2):
            continue

        X_train = X[train_idx]
        Y_train = Y[train_idx]
        x_test = X[i]
        y_true = Y[i]

        analog_pred = analog(X_train, Y_train, x_test, k=CONFIG["k"])

        mod = train(X_train, Y_train)
        reg_preds = reg(mod, x_test)

        all_preds = [analog_pred] + [reg_preds["ridge"], reg_preds["lasso"], reg_preds["elastic"], reg_preds["poisson"]]
        ensemble_pred = np.mean(np.vstack(all_preds), axis=0)

        climatology = climatemodel(Y_train)

        row = {"year": years[i]}

        for j, target in enumerate(CONFIG["b"]):
            row[f"{target}_actual"] = y_true[j]
            row[f"{target}_analog"] = analog_pred[j]
            row[f"{target}_ridge"] = reg_preds["ridge"][j]
            row[f"{target}_lasso"] = reg_preds["lasso"][j]
            row[f"{target}_elastic"] = reg_preds["elastic"][j]
            row[f"{target}_poisson"] = reg_preds["poisson"][j]
            row[f"{target}_ensemble"] = ensemble_pred[j]
            row[f"{target}_climo"] = climatology[j]

        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
hindcast_df = run_hindcast()
hindcast_df.head()

In [ ]:
def evaluate_hindcast(hindcast_df):
    rows = []

    for target in CONFIG["b"]:
        actual = hindcast_df[f"{target}_actual"].to_numpy()
        ensemble = hindcast_df[f"{target}_ensemble"].to_numpy()
        climo = hindcast_df[f"{target}_climo"].to_numpy()

        mae_model = mean_absolute_error(actual, ensemble)
        rmse_model = np.sqrt(mean_squared_error(actual, ensemble))

        mae_climo = mean_absolute_error(actual, climo)
        rmse_climo = np.sqrt(mean_squared_error(actual, climo))

        skill_rmse = 1 - (rmse_model / rmse_climo) if rmse_climo != 0 else np.nan
        corr = np.corrcoef(actual, ensemble)[0, 1] if len(actual) > 1 else np.nan
        bias = np.mean(ensemble - actual)

        rows.append({
            "Target": target,
            "Model_MAE": mae_model,
            "Model_RMSE": rmse_model,
            "Climo_MAE": mae_climo,
            "Climo_RMSE": rmse_climo,
            "RMSE_Skill_vs_Climo": skill_rmse,
            "Correlation": corr,
            "Bias": bias
        })

    return pd.DataFrame(rows)

In [ ]:
eval_df = evaluate_hindcast(hindcast_df)
eval_df

In [ ]:
def forecast_current():
    if not CONFIG["vals"]:
        raise ValueError("Fill in CONFIG['vals'] before running forecast_current().")

    x = np.array([CONFIG["vals"][p] for p in CONFIG["a"]], dtype=float)

    analog_pred, idx, dists = analog(X, Y, x, k=CONFIG["k"], return_analogs=True)

    print("\nTop Analog Years:")
    for i in range(len(idx)):
        print(f"{i+1}. Year {years[idx[i]]} | Distance = {dists[i]:.3f}")
    print()

    models = train(X, Y)
    reg_preds = reg(models, x)

    all_preds = [analog_pred, reg_preds["ridge"], reg_preds["lasso"], reg_preds["elastic"], reg_preds["poisson"]]
    ensemble_pred = np.mean(np.vstack(all_preds), axis=0)

    return pd.DataFrame({
        "Target": CONFIG["b"],
        "Analog": analog_pred,
        "Ridge": reg_preds["ridge"],
        "Lasso": reg_preds["lasso"],
        "ElasticNet": reg_preds["elastic"],
        "Poisson": reg_preds["poisson"],
        "Ensemble": ensemble_pred
    })

In [ ]:
forecast_current()